In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.utils.helper_functions import *
from src.pipeline.props_pipeline.ppm_pipeline import *
from src.pipeline.props_pipeline.apm_pipeline import *
from src.pipeline.props_pipeline.rpm_pipeline import *
from src.pipeline.props_pipeline.papm_pipeline import *
from src.pipeline.props_pipeline.min_pipeline import *
from src.live import *
from src.historical_analysis.dataScraper import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### Get updated lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
print("\nQuestionable Players:")
print(scraper.getQuestionablePlayers())
print("\nOut Players:")
print(scraper.getOutPlayers())
outPlayers = scraper.getOutPlayers()
scraper.updateTeamInfo()  # Update teamInfo.py


Questionable Players:
No data available. Run getDict() first.
{}

Out Players:
No data available. Run getDict() first.
{}
No data available. Run getDict() first.
No data available. Run getDict() first.


### Dataset

In [3]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
base_df = pd.concat([s25, s26])
base_df.tail()

,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,START_POSITION,pos,age
209,NaN,2025-26,201145,Jeff Green,Jeff,1610612745,HOU,Houston Rockets,22501194,2026-04-12T00:00:00,HOU vs. MEM,W,24.183333,2,8,0.250,0,4,0.00,2,2,1.000,2,3,5,1,3,1,1,0,2,1,6,7,16.5,0,0,16.0,1,24:11,1,103.9,105.5,105.5,86.1,91.1,91.1,17.8,14.4,14.4,0.059,0.33,7.7,0.063,0.100,0.081,23.1,23.3,0.250,0.338,0.174,0.181,114.13,110.16,91.80,110.16,0.016,55,2.0,8.0,48,105,0.457,14,49,0.286,22,30,0.733,21,43,64,23,11.0,10,8,6,10,24,132,31.0,122.0,128.2,94.9,98.1,27.1,30.1,0.479,2.09,14.9,0.452,0.821,0.627,0.107,0.524,0.558,107.3,103.0,85.83,103,0.642,1610612763,MEM,Memphis Grizzlies,39,95,0.411,13,49,0.265,10,10,1.000,6,31,37,23,13.0,8,6,8,24,10,101,-31.0,94.9,98.1,122.0,128.2,-27.1,-30.1,0.590,1.77,16.9,0.179,0.548,0.373,0.126,0.479,0.508,107.3,103.0,85.83,103,0.358,NaN,PF,39.0
210,NaN,2025-26,1631342,Daeqwon Plowden,Daeqwon,1610612758,SAC,Sacramento Kings,22501200,2026-04-12T00:00:00,SAC @ POR,L,22.200000,2,5,0.400,1,4,0.25,3,3,1.000,0,1,1,2,3,0,2,0,4,1,8,-8,15.2,0,0,16.0,1,22:12,1,118.2,114.6,114.6,127.2,134.0,134.0,-9.0,-19.5,-19.5,0.100,0.67,18.2,0.000,0.034,0.022,27.3,26.5,0.500,0.633,0.173,0.184,103.83,102.70,85.59,102.70,0.017,48,2.0,5.0,40,83,0.482,7,21,0.333,23,31,0.742,15,32,47,24,17.0,7,8,3,18,21,110,-12.0,111.5,112.2,120.1,123.2,-8.6,-11.0,0.600,1.41,17.3,0.370,0.600,0.495,0.173,0.524,0.569,100.1,98.5,82.08,98,0.478,1610612757,POR,Portland Trail Blazers,47,101,0.465,16,46,0.348,12,15,0.800,18,28,46,28,12.0,9,3,8,21,18,122,12.0,120.1,123.2,111.5,112.2,8.6,11.0,0.596,2.33,18.8,0.400,0.630,0.505,0.121,0.545,0.567,100.1,98.5,82.08,99,0.522,F,SG,27.0
211,NaN,2025-26,1630286,Trevon Scott,Trevon,1610612751,BKN,Brooklyn Nets,22501192,2026-04-12T00:00:00,BKN @ TOR,L,26.050000,3,8,0.375,2,5,0.40,0,0,0.000,1,2,3,1,2,1,0,0,2,0,8,-17,14.1,0,0,16.0,1,26:03,1,104.8,109.3,109.3,137.8,135.7,135.7,-32.9,-26.5,-26.5,0.053,0.50,9.1,0.038,0.125,0.071,18.2,18.2,0.500,0.500,0.164,0.161,102.67,101.34,84.45,101.34,0.027,54,3.0,8.0,35,86,0.407,9,34,0.265,22,29,0.759,13,25,38,21,16.0,9,0,8,27,20,101,-35.0,99.3,101.0,136.3,136.0,-37.1,-35.0,0.600,1.31,15.6,0.255,0.844,0.471,0.160,0.459,0.511,100.8,100.0,83.33,100,0.277,1610612761,TOR,Toronto Raptors,51,80,0.638,12,27,0.444,22,29,0.759,3,40,43,36,10.0,9,8,0,20,27,136,35.0,136.3,136.0,99.3,101.0,37.1,35.0,0.706,3.60,25.5,0.156,0.745,0.529,0.100,0.713,0.733,100.8,100.0,83.33,100,0.723,C,NaN,NaN
204,NaN,2025-26,1629020,

### Load latest odds on file

In [4]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')
if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_dds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_dds = pd.json_normalize(data)

print("Loaded:", file.name)
team_dds.head()

Loaded: NBA_20260413_141608.json


,home_team,away_team,commence_time,bookmakers
0,Charlotte Hornets,Miami Heat,2026-04-14 23:30:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
1,Phoenix Suns,Portland Trail Blazers,2026-04-15 02:10:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
2,Philadelphia 76ers,Orlando Magic,2026-04-15 23:30:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
3,Los Angeles Clippers,Golden State Warriors,2026-04-16 02:10:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
4,Cleveland Cavaliers,Toronto Raptors,2026-04-18 17:00:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."


In [5]:
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

#load season stats
pts_df = pd.read_csv('data/processed/training/S26_TRAINING_PPM.csv')
ast_df = pd.read_csv('data/processed/training/S26_TRAINING_APM.csv')
reb_df = pd.read_csv('data/processed/training/S26_TRAINING_RPM.csv')
min_df = pd.read_csv('data/processed/training/S26_TRAINING_MIN.csv')
pts_ast_df = pd.read_csv('data/processed/training/S26_TRAINING_PAPM.csv')

#load dfs lines
lines_dfs = pd.read_csv(dfs_file)
lines_dfs_pts = lines_dfs[(lines_dfs['CATEGORY'] == 'player_points')]
lines_dfs_ast = lines_dfs[(lines_dfs['CATEGORY'] == 'player_assists')]
lines_dfs_reb = lines_dfs[(lines_dfs['CATEGORY'] == 'player_rebounds')]
lines_dfs_pts_ast = lines_dfs[(lines_dfs['CATEGORY'] == 'player_points_assists')]
pts_names = lines_dfs_pts['NAME'].unique()
ast_names = lines_dfs_ast['NAME'].unique()
reb_names = lines_dfs_reb['NAME'].unique()
pts_ast_names = lines_dfs_pts_ast['NAME'].unique()

#load us lines with actual odds
lines_us = pd.read_csv(us_file)
lines_us_pts = lines_us[(lines_us['CATEGORY'] == 'player_points')]
lines_us_ast = lines_us[(lines_us['CATEGORY'] == 'player_assists')]
lines_us_reb = lines_us[(lines_us['CATEGORY'] == 'player_rebounds')]
lines_us_pts_ast = lines_us[(lines_us['CATEGORY'] == 'player_points_assists')]
print(f"DFS latest pull: {lines_dfs['DATA_PULLED_AT'].max()}")
print(f"US latest pull: {lines_us['DATA_PULLED_AT'].max()}")

lines_dfs_pts.head()

DFS latest pull: 2026-04-13 14:15:25
US latest pull: 2026-04-13 14:16:08


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,DraftKings Pick6,player_points,Bam Adebayo,Over,21.5,-137,2026-04-14,2026-04-13T21:15:08Z,2026-04-13 14:15:25
1,DraftKings Pick6,player_points,Bam Adebayo,Under,21.5,-137,2026-04-14,2026-04-13T21:15:08Z,2026-04-13 14:15:25
2,DraftKings Pick6,player_points,LaMelo Ball,Over,23.5,-137,2026-04-14,2026-04-13T21:15:08Z,2026-04-13 14:15:25
3,DraftKings Pick6,player_points,LaMelo Ball,Under,23.5,-137,2026-04-14,2026-04-13T21:15:08Z,2026-04-13 14:15:25
4,DraftKings Pick6,player_points,Andrew Wiggins,Over,14.5,-137,2026-04-14,2026-04-13T21:15:08Z,2026-04-13 14:15:25


### Load my models

In [6]:
import joblib

#minutes
min_bundle = joblib.load("src/models/saved_models/min_quantile_xgb.joblib")
min_quantile_models = min_bundle["quantile_models"]
min_feature_names = min_bundle["feature_names"]
min_scaler = min_bundle.get("scaler")

#points per minute
ppm_bundle = joblib.load("src/models/saved_models/ppm_quantile_xgb.joblib")
ppm_quantile_models = ppm_bundle["quantile_models"]
ppm_feature_names = ppm_bundle["feature_names"]
ppm_scaler = ppm_bundle.get("scaler")
#assists per minute
apm_bundle = joblib.load("src/models/saved_models/apm_quantile_xgb.joblib")
apm_quantile_models = apm_bundle["quantile_models"]
apm_feature_names = apm_bundle["feature_names"]
apm_scaler = apm_bundle.get("scaler")

#rebounds per minute
rpm_bundle = joblib.load("src/models/saved_models/rpm_quantile_xgb.joblib")
rpm_quantile_models = rpm_bundle["quantile_models"]
rpm_feature_names = rpm_bundle["feature_names"]
rpm_scaler = rpm_bundle.get("scaler")

#points per minute + assists per minute
# papm_bundle = joblib.load("src/models/saved_models/papm_quantile_xgb.joblib")
# papm_quantile_models = papm_bundle["quantile_models"]
# papm_feature_names = papm_bundle["feature_names"]
# papm_scaler = papm_bundle.get("scaler")

### Get Min predictions and Stat Per Min predictions 

In [7]:
pts_preds = predict_min_times_rate(
    pts_names, min_df, pts_df, current_date,
    name_dict=nameDict,
    rate_pipeline=ppm_pipeline,
    rate_quantile_models=ppm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="PTS",
)
ast_preds = predict_min_times_rate(
    ast_names, min_df, ast_df, current_date,
    name_dict=nameDict,
    rate_pipeline=apm_pipeline,
    rate_quantile_models=apm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="AST",
)
reb_preds = predict_min_times_rate(
    reb_names, min_df, reb_df, current_date,
    name_dict=nameDict,
    rate_pipeline=rpm_pipeline,
    rate_quantile_models=rpm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="REB",
)
# pts_ast_preds = predict_min_times_rate(
#     pts_ast_names, min_df, pts_ast_df, current_date,
#     name_dict=nameDict,
#     rate_pipeline=papm_pipeline,
#     rate_quantile_models=papm_quantile_models,
#     min_quantile_models=min_quantile_models,
#     stat_prefix="PTS+AST",
# )
ast_preds.head(10)

[SKIP] Kelly Oubre Jr: single positional indexer is out-of-bounds
[SKIP] Wendell Carter Jr: single positional indexer is out-of-bounds
[SKIP] Derrick Jones: single positional indexer is out-of-bounds
No game found for CLE within 3 days from 2026-04-13
[SKIP] Donovan Mitchell: 'float' object has no attribute 'round'
No game found for TOR within 3 days from 2026-04-13
[SKIP] Brandon Ingram: 'float' object has no attribute 'round'
No game found for CLE within 3 days from 2026-04-13
[SKIP] James Harden: 'float' object has no attribute 'round'
[SKIP] R.J. Barrett: single positional indexer is out-of-bounds
No game found for CLE within 3 days from 2026-04-13
[SKIP] Evan Mobley: 'float' object has no attribute 'round'
No game found for TOR within 3 days from 2026-04-13
[SKIP] Scottie Barnes: 'float' object has no attribute 'round'
No game found for CLE within 3 days from 2026-04-13
[SKIP] Jarrett Allen: 'float' object has no attribute 'round'
No game found for TOR within 3 days from 2026-04-1

,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,STAT_Q10,STAT_Q50,STAT_Q90,RATE_HISTORY
0,LaMelo Ball,AST,6.37,29.54,37.33,0.1281,0.2597,0.3919,0.82,7.67,14.63,"[0.163291966035271, 0.091324200913242, 0.39239..."
1,Miles Bridges,AST,9.87,26.48,35.92,0.0213,0.1073,0.2040,0.21,2.84,7.33,"[0.159846547314578, 0.0591366055588409, 0.0636..."
2,Bam Adebayo,AST,3.36,33.94,40.42,0.0458,0.1330,0.2425,0.15,4.51,9.80,"[0.071599045346062, 0.057175528873642, 0.02598..."
3,Andrew Wiggins,AST,10.92,27.77,38.30,0.0003,0.0946,0.1756,0.00,2.63,6.72,"[0.1733102253032928, 0.056657223796034, 0.1289..."
4,Scoot Henderson,AST,9.15,30.07,38.57,0.0449,0.1695,0.2781,0.41,5.10,10.73,"[0.1002673796791443, 0.0824062628759785, 0.201..."
5,Deni Avdija,AST,4.73,33.31,40.90,0.0759,0.1804,0.2932,0.36,6.01,11.99,"[0.2486016159105034, 0.2355404344412457, 0.148..."
6,Devin Booker,AST,-2.20,26.68,39.95,0.0916,0.1834,0.3035,-0.20,4.89,12.12,"[0.1442169022209402, 0.0, 0.160513643659711, 0..."
7,Jalen Green,AST,1.61,21.19,34.41,0.0273,0.1127,0.2136,0.04,2.39,7.35,"[0.1164144353899883, 0.1133465570983281, 0.029..."
8,Davion Mitchell,AST,9.62,29.19,38.51,0.0836,0.2027,0.3192,0.80,5.92,12.29,"[0.2074688796680497, 0.18796992481203, 0.15686..."
9,Tyler Herro,AST,5.75,32.29,39.81,0.0437,0.1452,0.2477,0.25,4.69,9.86,"[0.1608751608751608, 0.2410929547281007, 0.099..."


### Get Line Probabilities

In [8]:
all_line_probs = pd.concat([
    line_probs_for_market(ast_preds, lines_dfs_ast, nameDict, run_pts_simulation),
    line_probs_for_market(reb_preds, lines_dfs_reb, nameDict, run_pts_simulation),
    line_probs_for_market(pts_preds, lines_dfs_pts, nameDict, run_pts_simulation),
], ignore_index=True)
all_line_probs.sample(10)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER
16,Darius Garland,AST,6.5,6.76,31.09,40.74,0.66,6.96,14.27,0.410,0.590
45,Ryan Kalkbrenner,REB,4.5,10.34,17.02,22.27,0.97,4.11,9.48,0.367,0.633
12,Grayson Allen,AST,2.5,9.33,22.15,33.50,0.02,2.26,6.90,0.453,0.547
48,Draymond Green,REB,5.5,10.96,22.67,33.32,0.96,4.60,11.23,0.338,0.662
90,Tristan da Silva,PTS,6.5,13.44,22.24,30.98,1.45,7.63,21.34,0.681,0.319
82,Tyrese Maxey,PTS,29.5,4.02,34.00,42.90,1.59,23.07,48.18,0.226,0.774
33,Donovan Clingan,REB,12.5,9.63,23.20,32.37,2.08,9.40,18.62,0.286,0.714
29,Pelle Larsson,REB,3.5,10.92,25.67,35.17,0.22,2.84,7.71,0.399,0.602
1,Miles Bridges,AST,2.5,9.87,26.48,35.92,0.21,2.84,7.33,0.524,0.476
104,Sion James,PTS,3.5,9.43,17.26,25.40,0.22,4.81,18.32,0.575,0.425


In [31]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='Underdog',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

underdog_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
underdog_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
1,Tyler Herro,AST,4.5,14.25,31.05,39.83,0.62,4.56,9.73,0.447,0.553,AST,Underdog,Atlanta Hawks,-4.0,243.0,112.6,9.0,102.48,6.0,-105.0,-123.0,0.512,0.552,4.4,4.0,1.43,-0.1,-0.5,0.070,0.472,0.528,-7.85,-4.27,0.4,0.4,0.53,0.54,33.59,5.37,0.23,0.04,6.00,5.0
115,Immanuel Quickley,PTS,13.5,11.06,21.82,33.02,1.71,10.35,28.39,0.258,0.742,PTS,Underdog,Brooklyn Nets,-22.0,219.0,117.8,25.0,97.59,27.0,-137.0,-137.0,0.578,0.578,12.0,11.0,6.06,-1.5,-2.5,0.248,0.402,0.598,-30.46,3.45,0.2,0.3,0.47,0.68,29.63,6.28,0.17,0.04,18.75,4.0
16,LeBron James,AST,10.5,12.65,31.94,41.05,1.70,8.13,16.25,0.310,0.690,AST,Underdog,Utah Jazz,-14.5,236.5,120.8,29.0,103.52,2.0,-137.0,-137.0,0.578,0.578,8.9,9.5,3.98,-1.6,-1.0,0.402,0.344,0.656,-40.49,13.48,0.6,0.4,0.27,0.19,33.73,3.96,0.24,0.06,10.14,7.0
172,Deni Avdija,PTS,25.5,15.45,32.27,40.29,5.79,21.30,42.47,0.398,0.602,PTS,Underdog,Sacramento Kings,-16.5,228.5,120.2,28.0,100.14,17.0,-137.0,-137.0,0.578,0.578,24.3,24.5,5.52,-1.2,-1.0,0.217,0.414,0.586,-28.38,1.37,1.0,0.5,0.40,0.31,32.76,6.24,0.30,0.02,21.57,7.0
171,Jrue Holiday,PTS,15.5,13.78,30.06,39.31,3.64,15.69,34.41,0.444,0.556,PTS,Underdog,Sacramento Kings,-16.5,228.5,120.2,28.0,100.14,17.0,-105.0,-123.0,0.512,0.552,16.5,13.0,7.74,1.0,-2.5,-0.129,0.551,0.449,7.58,-18.60,0.6,0.4,0.33,0.34,31.02,5.27,0.22,0.06,5.00,2.0


In [32]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='PrizePicks',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

prizePicks_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
prizePicks_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
181,Ethan Thompson,PTS,11.5,23.69,32.65,38.45,3.96,12.31,29.50,0.608,0.392,PTS,PrizePicks,Detroit Pistons,13.5,229.0,108.8,2.0,99.82,19.0,-137.0,-137.0,0.578,0.578,11.0,13.0,7.44,-0.5,1.5,0.067,0.473,0.527,-18.17,-8.83,0.6,0.5,0.33,0.23,28.95,7.70,0.16,0.05,10.00,1.0
148,Julian Champagnie,PTS,12.5,14.88,25.00,35.40,2.59,9.73,26.68,0.434,0.566,PTS,PrizePicks,Denver Nuggets,-11.0,233.2,116.0,21.0,99.46,20.0,-137.0,-137.0,0.578,0.578,10.9,13.0,5.24,-1.1,1.0,0.210,0.417,0.583,-27.86,0.85,0.6,0.6,0.53,0.37,26.49,3.01,0.15,0.06,15.00,6.0
176,Devin Carter,PTS,16.5,23.42,30.76,36.83,5.82,16.70,36.20,0.543,0.457,PTS,PrizePicks,Portland Trail Blazers,16.5,228.5,113.6,11.0,101.67,9.0,-137.0,-137.0,0.578,0.578,14.8,15.0,7.87,-1.7,-1.5,0.216,0.414,0.586,-28.38,1.37,0.4,0.4,0.33,0.10,27.91,6.36,0.23,0.05,0.00,2.0
14,Reed Sheppard,AST,4.5,15.08,25.96,35.13,0.29,3.55,8.37,0.313,0.687,AST,PrizePicks,Memphis Grizzlies,-13.0,225.5,118.3,27.0,101.68,8.0,-137.0,-137.0,0.578,0.578,3.2,2.5,2.10,-1.3,-2.0,0.619,0.268,0.732,-53.64,26.63,0.2,0.3,0.33,0.17,25.45,5.76,0.20,0.06,2.00,4.0
125,Duncan Robinson,PTS,9.5,12.99,24.69,34.33,2.81,10.86,29.18,0.805,0.195,PTS,PrizePicks,Indiana Pacers,-13.5,229.0,117.7,24.0,101.74,7.0,-137.0,-137.0,0.578,0.578,14.8,14.5,3.77,4.8,4.5,-1.273,0.898,0.102,55.35,-82.35,1.0,0.9,0.80,0.55,27.10,2.67,0.17,0.04,13.14,7.0


In [33]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='Betr DFS',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

betr_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
betr_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
106,Pelle Larsson,PTS,12.5,23.20,31.84,38.22,4.93,13.49,31.11,0.658,0.342,PTS,Betr DFS,Atlanta Hawks,-4.0,243.0,112.6,9.0,102.48,6.0,-137.0,-137.0,0.578,0.578,14.1,14.5,5.65,1.6,2.0,-0.283,0.611,0.389,5.70,-32.71,0.6,0.6,0.67,0.29,30.13,5.73,0.19,0.05,9.00,6.0
173,Toumani Camara,PTS,14.5,15.93,30.85,40.47,3.07,13.16,28.80,0.503,0.497,PTS,Betr DFS,Sacramento Kings,-16.5,228.5,120.2,28.0,100.14,17.0,-114.0,-114.0,0.533,0.533,18.5,17.5,9.54,4.0,3.0,-0.419,0.662,0.338,24.27,-36.55,0.8,0.7,0.60,0.36,32.93,5.32,0.18,0.05,10.29,7.0
130,VJ Edgecombe,PTS,18.5,28.12,36.95,41.83,6.97,17.34,29.80,0.427,0.573,PTS,Betr DFS,Milwaukee Bucks,-15.0,227.0,118.3,26.0,98.29,23.0,-102.0,-127.0,0.505,0.559,18.3,17.5,7.48,-0.2,-1.0,0.027,0.489,0.511,-3.16,-8.66,0.4,0.5,0.47,0.35,35.52,4.46,0.20,0.06,12.00,3.0
181,Ethan Thompson,PTS,11.5,23.69,32.65,38.45,3.96,12.31,29.50,0.608,0.392,PTS,Betr DFS,Detroit Pistons,13.5,229.0,108.8,2.0,99.82,19.0,-137.0,-137.0,0.578,0.578,11.0,13.0,7.44,-0.5,1.5,0.067,0.473,0.527,-18.17,-8.83,0.6,0.5,0.33,0.23,28.95,7.70,0.16,0.05,10.00,1.0
159,Brice Sensabaugh,PTS,21.5,13.46,26.61,36.49,3.39,16.25,38.45,0.402,0.598,PTS,Betr DFS,Los Angeles Lakers,14.5,236.5,115.7,20.0,99.14,22.0,-137.0,-137.0,0.578,0.578,24.4,23.0,8.30,2.9,1.5,-0.349,0.636,0.364,10.02,-37.03,0.4,0.6,0.53,0.15,31.19,6.22,0.29,0.06,5.33,6.0


In [34]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='DraftKings Pick6',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

draftKings_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
draftKings_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
71,Donovan Clingan,REB,12.5,13.49,24.16,34.35,3.06,9.59,19.81,0.342,0.658,REB,DraftKings Pick6,Sacramento Kings,-16.5,228.5,120.2,28.0,100.14,17.0,104.0,-135.0,0.490,0.574,11.4,12.5,4.27,-1.1,0.0,0.258,0.398,0.602,-18.81,4.79,0.2,0.5,0.53,0.29,26.44,3.52,0.16,0.05,9.17,6.0
131,Tyrese Maxey,PTS,28.5,14.71,34.07,43.64,5.92,23.66,47.78,0.296,0.704,PTS,DraftKings Pick6,Milwaukee Bucks,-15.0,227.0,118.3,26.0,98.29,23.0,-105.0,-123.0,0.512,0.552,24.8,24.5,4.94,-3.7,-4.0,0.749,0.227,0.773,-55.68,40.15,0.2,0.2,0.33,0.47,37.62,3.84,0.27,0.04,31.50,6.0
130,VJ Edgecombe,PTS,18.5,28.12,36.95,41.83,6.97,17.34,29.80,0.427,0.573,PTS,DraftKings Pick6,Milwaukee Bucks,-15.0,227.0,118.3,26.0,98.29,23.0,-102.0,-127.0,0.505,0.559,18.3,17.5,7.48,-0.2,-1.0,0.027,0.489,0.511,-3.16,-8.66,0.4,0.5,0.47,0.35,35.52,4.46,0.20,0.06,12.00,3.0
195,Donovan Clingan,PTS,13.5,13.49,24.16,34.35,3.11,10.06,25.22,0.377,0.623,PTS,DraftKings Pick6,Sacramento Kings,-16.5,228.5,120.2,28.0,100.14,17.0,100.0,-130.0,0.500,0.565,10.6,9.0,5.95,-2.9,-4.5,0.487,0.313,0.687,-37.40,21.55,0.4,0.4,0.53,0.26,26.44,3.52,0.16,0.05,9.67,6.0
91,Quenton Jackson,REB,3.5,20.60,28.36,34.81,0.74,3.38,7.56,0.544,0.456,REB,DraftKings Pick6,Detroit Pistons,13.5,229.0,108.8,2.0,99.82,19.0,113.0,-147.0,0.469,0.595,3.1,3.0,1.79,-0.4,-0.5,0.223,0.412,0.588,-12.24,-1.20,0.6,0.4,0.33,0.22,20.51,7.12,0.19,0.06,1.50,2.0


In [35]:
all_line_probs = pd.concat([underdog_all_lines, prizePicks_all_lines, betr_all_lines, draftKings_all_lines])
all_line_probs.to_json('data/props/ev_analysis/all_line_probs.json', orient='records', lines=True)
all_line_probs.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
110,Jalen Suggs,PTS,12.5,12.82,27.20,36.19,2.61,12.23,29.09,0.471,0.529,PTS,Underdog,Boston Celtics,-12.0,217.5,111.8,4.0,95.46,30.0,-137.0,-137.0,0.578,0.578,12.7,12.0,4.16,0.2,-0.5,-0.048,0.519,0.481,-10.22,-16.79,0.2,0.4,0.47,0.55,30.20,5.54,0.20,0.06,16.67,3.0
165,Mark Williams,PTS,10.5,10.08,22.86,32.97,2.91,12.46,31.71,0.569,0.431,PTS,Betr DFS,Oklahoma City Thunder,5.5,213.5,106.1,1.0,100.40,15.0,-137.0,-137.0,0.578,0.578,9.4,9.5,4.99,-1.1,-1.0,0.220,0.413,0.587,-28.55,1.55,0.6,0.4,0.53,0.62,21.45,5.06,0.18,0.04,9.60,5.0
195,Donovan Clingan,PTS,13.5,13.49,24.16,34.35,3.11,10.06,25.22,0.377,0.623,PTS,Betr DFS,Sacramento Kings,-16.5,228.5,120.2,28.0,100.14,17.0,-137.0,-137.0,0.578,0.578,10.6,9.0,5.95,-1.9,-3.5,0.319,0.375,0.625,-35.13,8.12,0.4,0.4,0.53,0.29,26.44,3.52,0.16,0.05,9.67,6.0
100,Tyler Herro,PTS,21.5,14.25,31.05,39.83,4.62,18.09,39.64,0.323,0.677,PTS,PrizePicks,Atlanta Hawks,-4.0,243.0,112.6,9.0,102.48,6.0,-137.0,-137.0,0.578,0.578,20.1,18.0,6.67,-1.4,-3.5,0.210,0.417,0.583,-27.86,0.85,0.4,0.3,0.40,0.58,33.59,5.37,0.23,0.04,24.60,5.0
105,Nickeil Alexander-Walker,PTS,23.5,17.16,31.52,40.84,4.11,15.68,39.25,0.345,0.654,PTS,Underdog,Miami Heat,4.0,243.0,113.7,13.0,104.22,1.0,-137.0,-137.0,0.578,0.578,24.3,23.0,6.15,0.8,-0.5,-0.130,0.552,0.448,-4.51,-22.50,0.6,0.5,0.40,0.14,35.34,4.38,0.21,0.03,15.60,5.0


### Get top EVs for 2 legs

In [36]:
slate_path = build_greedy_slate(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks.json",
)
print(slate_path)

Legs: 108  |  Pairs: 262  |  Slate: 6  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json


In [37]:
slate_path = build_greedy_slate(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog.json",
)
print(slate_path)

Legs: 88  |  Pairs: 208  |  Slate: 5  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json


In [38]:
slate_path = build_greedy_slate(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings.json",
)
print(slate_path)

Legs: 13  |  Pairs: 11  |  Slate: 2  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings.json


In [39]:
slate_path = build_greedy_slate(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr.json",
)
print(slate_path)

Legs: 83  |  Pairs: 213  |  Slate: 6  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json


### Top EVs for 3 Legs

In [40]:
slate_path = build_greedy_slate_3leg(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks_3leg.json",
)
print(slate_path)

Legs: 108  |  Triples: 5781  |  Slate: 5  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks_3leg.json


In [41]:
slate_path = build_greedy_slate_3leg(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog_3leg.json",
)
print(slate_path)

Legs: 88  |  Triples: 3797  |  Slate: 4  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog_3leg.json


In [42]:
slate_path = build_greedy_slate_3leg(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr_3leg.json",
)
print(slate_path)

Legs: 83  |  Triples: 4005  |  Slate: 5  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr_3leg.json


In [43]:
slate_path = build_greedy_slate_3leg(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings_3leg.json",
)
print(slate_path)

Legs: 13  |  Triples: 29  |  Slate: 2  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings_3leg.json
